#STURM-Flood

In [10]:
#@title Imports

import random
import os
import sys
import zipfile
import ee

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [11]:
#@title Google auth + Mount Drive + Optional clone

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
clone_repo = False  #@param {type:"boolean"}
reset_export = True  #@param {type:"boolean"}
gee_project = "356457881639"  #@param {type:"string"}

# Always authenticate + mount
auth.authenticate_user()
drive.mount(mount_point)

ee.Initialize(project=gee_project)

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

if clone_repo:
    project_root = os.path.join(root_path, "STURM-fusion")
    if not os.path.exists(project_root):
        !git clone {repo_url} "{project_root}"
else:
    project_root = root_path

sys.path.append(project_root)

from config.config import CFG
cfg = CFG()
cfg.ROOT = Path(project_root)
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive"
cfg.GEE_PROJECT = gee_project

print("✅ ROOT:", cfg.ROOT)
print("✅ DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("✅ GEE project:", cfg.GEE_PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ROOT: /content/drive/MyDrive/MSc/STURM-fusion
✅ DRIVE_ROOT: /content/drive/MyDrive
✅ GEE project: 356457881639


In [12]:
from src.data.sturm_flood import *
from src.pipeline.build_dataset import *
from src.utils.io import *
from src.gee.tasks import has_active_tasks
from src.huggingface.push_dataset import push_zip_to_hf

In [ ]:
if reset_export:
    clear_export_folder(cfg)

create_dataset_structure(cfg)

In [ ]:
#@title Download Dataset

download_and_extract(cfg)

In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Starting image collection... 🚀"  )
    images, df_fusion = process_csv(cfg.OLD_S2_METADATA_CSV, cfg)
    print(len(images), "images processed.")
else:
   print("Active GEE tasks detected. Please wait for them to finish starting Image collection. ⏳")


In [ ]:
save_dataframe_to_csv(df_fusion, cfg.NEW_METADATA_CSV)

In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Starting export... 🚀"  )
    export_all_s1_images(images, cfg)
else:
    print("Active GEE tasks detected. Please wait for them to finish before exporting. ⏳")

In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Creating Dataset... 🚀"  )
    assemble_dataset(cfg)
else:
    print("Active GEE tasks detected. Please wait for them to finish before creating dataset. ⏳")

In [ ]:
#preprocessing_s1_pipeline(cfg.NEW_S1_PATH)

In [ ]:
import rasterio
from pathlib import Path


def check_image_shapes(dir1, dir2):
    """
    Compares image shapes between two directories (e.g. S1 vs S2)

    Assumes matching filenames.
    """

    dir1 = Path(dir1)
    dir2 = Path(dir2)

    mismatches = []
    missing = []

    files1 = {f.name for f in dir1.glob("*.tif")}

    for fname in files1:
        path1 = dir1 / fname
        path2 = dir2 / fname

        if not path2.exists():
            missing.append(fname)
            continue

        with rasterio.open(path1) as src1, rasterio.open(path2) as src2:
            shape1 = (src1.height, src1.width)
            shape2 = (src2.height, src2.width)

        if shape1 != shape2:
            mismatches.append((fname, shape1, shape2))

    print(f"\n📊 Shape Check Results:")
    print(f"Checked: {len(files1)} files")
    print(f"Missing in dir2: {len(missing)}")
    print(f"Mismatched shapes: {len(mismatches)}")

    return {
        "missing": missing,
        "mismatches": mismatches
    }

result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)


📊 Shape Check Results:
Checked: 1970 files
Missing in dir2: 0
Mismatched shapes: 0


In [ ]:
import numpy as np

tif_path = cfg.NEW_S1_PATH / "EMSR279_04TUDELA_07_10_2_1.tif"  # Replace with actual filename

def get_band_min_max(dir_path):
    dir_path = Path(dir_path)

    band_mins = None
    band_maxs = None

    for tif_path in dir_path.glob("*.tif"):
        with rasterio.open(tif_path) as src:
            data = src.read()  # [C, H, W]

            if band_mins is None:
                band_mins = np.full(data.shape[0], np.inf)
                band_maxs = np.full(data.shape[0], -np.inf)

            for b in range(data.shape[0]):
                band_mins[b] = min(band_mins[b], np.nanmin(data[b]))
                band_maxs[b] = max(band_maxs[b], np.nanmax(data[b]))

    print("\n📊 Per-band stats:")
    for i, (mn, mx) in enumerate(zip(band_mins, band_maxs)):
        print(f"Band {i+1}: min={mn:.3f}, max={mx:.3f}")

    return band_mins, band_maxs

from pathlib import Path
import rasterio
import numpy as np

def get_band_percentiles(dir_path, percentiles=[1, 5, 50, 95, 99]):
    dir_path = Path(dir_path)

    band_values = None

    for tif_path in dir_path.glob("*.tif"):
        with rasterio.open(tif_path) as src:
            data = src.read()  # [C, H, W]

            if band_values is None:
                band_values = [[] for _ in range(data.shape[0])]

            for b in range(data.shape[0]):
                # flatten and remove NaNs
                vals = data[b].ravel()
                vals = vals[~np.isnan(vals)]

                band_values[b].append(vals)

    print("\n📊 Per-band percentiles:")

    results = {}

    for b, vals_list in enumerate(band_values):
        all_vals = np.concatenate(vals_list)

        p_vals = np.percentile(all_vals, percentiles)

        results[b] = dict(zip(percentiles, p_vals))

        print(f"\nBand {b+1}:")
        for p, v in zip(percentiles, p_vals):
            print(f"  P{p}: {v:.3f}")

    return results

get_band_min_max(cfg.NEW_S1_PATH)
get_band_percentiles(cfg.NEW_S1_PATH)

/tmp/ipykernel_3479/1725855983.py:20: RuntimeWarning: All-NaN slice encountered
  band_mins[b] = min(band_mins[b], np.nanmin(data[b]))
/tmp/ipykernel_3479/1725855983.py:21: RuntimeWarning: All-NaN slice encountered
  band_maxs[b] = max(band_maxs[b], np.nanmax(data[b]))



📊 Per-band stats:
Band 1: min=-46.106, max=39.017
Band 2: min=-55.722, max=22.667

📊 Per-band percentiles:

Band 1:
  P1: -25.988
  P5: -23.240
  P50: -10.727
  P95: -6.101
  P99: -3.751

Band 2:
  P1: -30.512
  P5: -28.380
  P50: -17.183
  P95: -12.697
  P99: -11.029


{0: {1: np.float64(-25.987531776428224),
  5: np.float64(-23.24048614501953),
  50: np.float64(-10.726836204528809),
  95: np.float64(-6.101193118095399),
  99: np.float64(-3.751161069869994)},
 1: {1: np.float64(-30.51234727859497),
  5: np.float64(-28.379851627349854),
  50: np.float64(-17.18290901184082),
  95: np.float64(-12.69680728912354),
  99: np.float64(-11.02862907409667)}}

In [ ]:
import pandas as pd


def get_max_time_difference_with_row(csv_path, sentinel_timestamp):
    df = pd.read_csv(csv_path)

    df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")
    df[sentinel_timestamp] = pd.to_datetime(df[sentinel_timestamp], dayfirst=True, errors="coerce")

    df = df.dropna(subset=["floodmap_date", sentinel_timestamp])

    df["time_diff_hours"] = (
        (df[sentinel_timestamp] - df["floodmap_date"])
        .abs()
        .dt.total_seconds() / 3600
    )

    idx = df["time_diff_hours"].idxmax()
    row = df.loc[idx]

    print(f"⏱️ Max time difference: {row['time_diff_hours']:.2f} hours")
    print(row)

    return row

max_s2_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel2_timestamp")

print(f"Max S2 time difference: {max_s2_diff['time_diff_hours']:.2f} hours")

max_s1_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel1_timestamp")

print(f"Max S1 time difference: {max_s1_diff['time_diff_hours']:.2f} hours")

⏱️ Max time difference: 33.86 hours
Unnamed: 0                                    1138
ems_code                                   EMSR470
aoi_code                                     AOI01
floodmap_id              EMSR470_AOI01_DEL_PRODUCT
event_type                             Flash flood
country                                       Togo
tile_id                EMSR470_AOI01_04_27_1_2.tif
floodmap_date                  2020-10-14 20:11:00
sentinel2_timestamp            2020-10-13 10:19:09
sentinel1_timestamp               14/10/2020 18:11
time_diff_hours                          33.864167
Name: 527, dtype: object
Max S2 time difference: 33.86 hours
⏱️ Max time difference: 22.83 hours
Unnamed: 0                                     721
ems_code                                   EMSR424
aoi_code                                     AOI01
floodmap_id              EMSR424_AOI01_DEL_PRODUCT
event_type                          Riverine flood
country                                 Madagascar


/tmp/ipykernel_3479/2481064471.py:7: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")
/tmp/ipykernel_3479/2481064471.py:8: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[sentinel_timestamp] = pd.to_datetime(df[sentinel_timestamp], dayfirst=True, errors="coerce")
/tmp/ipykernel_3479/2481064471.py:7: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")


In [ ]:
zip_dataset(cfg)

🗜️ Zipping dataset directly...
✅ Dataset zipped at: /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset.zip


PosixPath('/content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset.zip')

In [16]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN)

url = push_zip_to_hf(
    zip_path=cfg.NEW_ZIP_PATH,
    repo_id=cfg.HF_REPO_ID,
    path_in_repo="Dataset.zip",
    private=False,
)
print(url)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...URM-fusion-24/Dataset.zip:   0%|          | 1.05MB / 1.30GB            

https://huggingface.co/datasets/tax2310/STURM-fusion-24/commit/61092679ba80135112679e90b6c38331e064b7c9


In [ ]:
def cancel_all_tasks(verbose=True):
    """
    Cancels all READY or RUNNING Earth Engine tasks.
    """

    tasks = ee.batch.Task.list()

    cancelled = 0
    skipped = 0

    for t in tasks:
        status = t.status()
        state = status['state']
        desc = status.get('description', 'N/A')

        if state in ['READY', 'RUNNING']:
            t.cancel()
            cancelled += 1
            if verbose:
                print(f"❌ Cancelled: {desc} ({state})")
        else:
            skipped += 1
            if verbose:
                print(f"⏭️ Skipped: {desc} ({state})")

    print("\n📊 Summary:")
    print(f"Cancelled: {cancelled}")
    print(f"Skipped (already done): {skipped}")

#cancel_all_tasks()